In [2]:
# Imports
from pathlib import Path
from experiment.utils import TrainedModel, TrainedModelID

import pandas as pd
import torch
from neuralhydrology.nh_run import start_run, eval_run, finetune
from experiment.eval import evaluate_models

In [3]:
model = TrainedModel(TrainedModelID.SOTA_20)

cutoff = 0.3
df = pd.read_csv(model.metrics_file, dtype={'basin':str})

basin_data = df.loc[df['NSE'] < cutoff].sample(n=1)

basin = basin_data.basin.iloc[0]

basin

'08165300'

In [4]:
# Add the path to the pre-trained model to the finetune config
with open("finetune.yml", "a") as fp:
    fp.write(f"\nbase_run_dir: {model.run_dir.absolute()}")
    
# Create a basin file with the basin we selected above
with open("finetune_basin.txt", "w") as fp:
    fp.write(basin)

In [5]:
# finetune
finetune(Path("finetune.yml"))

DuplicateKeyError: while constructing a mapping
  in "finetune.yml", line 1, column 1
found duplicate key "base_run_dir" with value "/home/admin/Fine-Flood-Forecasts/experiment/models/runs/sota_20" (original value: "/home/admin/Fine-Flood-Forecasts/experiment/models/runs/sota_20")
  in "finetune.yml", line 17, column 1

To suppress this check see:
    http://yaml.readthedocs.io/en/latest/api.html#duplicate-keys


In [ ]:
config_file_path = Path(__file__).parent / 'runs' / 'cudalstm_531_basins_finetuned' / 'config.yml'
finetuned_model = TrainedModel(config_file_path_or_experiment_name=config_file_path)
evaluate_models([model, finetuned_model], basins=[basin])

In [ ]:
# load test results of the base run
df_pretrained = pd.read_csv(run_dir / "test/model_epoch003/test_metrics.csv", dtype={'basin': str})
df_pretrained = df_pretrained.set_index("basin")
    
# load test results of the finetuned model
df_finetuned = pd.read_csv(finetune_dir / "test/model_epoch010/test_metrics.csv", dtype={'basin': str})
df_finetuned = df_finetuned.set_index("basin")
    
# extract basin performance
base_model_nse = df_pretrained.loc[df_pretrained.index == basin, "NSE"].values[0]
finetune_nse = df_finetuned.loc[df_finetuned.index == basin, "NSE"].values[0]
print(f"Basin {basin} base model performance: {base_model_nse:.3f}")
print(f"Performance after finetuning: {finetune_nse:.3f}")

So we see roughly the same performance increase in the test period (slightly higher), which is great. However, note that a) our base model was not optimally trained (we stopped quite early) but also b) the finetuning settings were chosen rather randomly. From our experience so far, you can almost always get performance increases for individual basins with finetuning, but it is difficult to find settings that are universally applicable. However, this tutorial was just a showcase of how easy it actually is to finetune models with the NeuralHydrology library. Now it is up to you to experiment with it.